# Musanga V4 — NBAA: Annotation Consistency — exploratory audit

**Purpose:** explore *two distinct annotation tasks* before modelling: (1) five experts' `Musanga / non-Musanga / uncertain` classifications on the same candidate crowns, and (2) independently digitized Musanga-positive crown polygons in six verification circles by **four** annotators. Polygon absence is **not** automatically a non-Musanga vote.

**Scientific safeguards:** exact `verify_id` joins; no ground-truth claims or species inference; no models trained; no changes to V3_1/V4 reference or folds; spatial coordinates reprojected to UTM 35N before area/overlap calculations; independent polygon sets are *not* matched by `crown_id` (local per-annotator IDs). Comparisons of polygon overlap are restricted to circles both experts annotated. Shape uncertainty and annotation completeness are flagged, not silently resolved.

Read-only source: `Final/Data/Annotations/Verification/`. All outputs: `Final/Outputs/v4/NBAA_annotation_consistency/`.

**Run cells sequentially in Colab.** Large mosaics and LAS files are not accessed. Sections 00–11 are exploratory; Sections 12–14 extract frozen image features and fit simple annotator-specific classifiers.

## 00 — V3_1-compatible Drive setup and file inventory

In [ ]:
%pip -q install geopandas shapely pyogrio scipy scikit-learn pandas matplotlib
from pathlib import Path
import json, re, math, warnings, itertools
from collections import Counter, defaultdict
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from shapely.geometry import shape
from shapely.ops import unary_union
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import cohen_kappa_score
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                         Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None,'Original Musanga Final directory unavailable'
DATA=ROOT/'Data'
SRC=DATA/'Annotations'/'Verification'
assert SRC.is_dir(),f'Expected input directory missing: {SRC}'
OUT=ROOT/'Outputs/v4/NBAA_annotation_consistency'
OUT.mkdir(parents=True,exist_ok=True)
files=sorted(p for p in SRC.iterdir() if p.is_file())
inventory=pd.DataFrame([dict(filename=p.name,extension=p.suffix.lower() or '(none)',
                             size_bytes=p.stat().st_size) for p in files])
inventory.to_csv(OUT/'NBAA_00_file_inventory.csv',index=False)
display(inventory)
print('Read-only source:',SRC)
print('Write-only NBAA output:',OUT)
print('Reference V3_1 / V4 datasets are NOT edited.')

## 01 — Classification CSV audit: exact crown IDs, labels and missingness

In [ ]:
CLASS_CSV=sorted(SRC.glob('musanga_classifications_*.csv'))
assert len(CLASS_CSV)==5, f'Expected 5 independent CSVs, found {[p.name for p in CLASS_CSV]}'
def who(p):
    n=p.name.casefold()
    if 'nadine' in n:return 'Nadine'
    if 'marjin' in n or 'marijn' in n:return 'Marijn'
    if 'joseph' in n:return 'Joseph'
    if 'thomas' in n:return 'Thomas'
    if 'kristof' in n:return 'Kristof'
    raise ValueError('Unknown annotator '+str(p))
NORM={'musanga':'M','non_musanga':'N','non-musanga':'N','non musanga':'N','uncertain':'U',
      'm':'M','n':'N','u':'U'}
raw_tables={};qa_rows=[];table_rows=[]
for p in CLASS_CSV:
    person=who(p);d=pd.read_csv(p,dtype={'verify_id':str})
    required={'verify_id','circle_id','classification'}
    assert required.issubset(d.columns),(p.name,required-set(d.columns))
    d['verify_id']=d.verify_id.astype(str).str.strip()
    d['label']=d.classification.astype(str).str.strip().str.casefold().map(NORM)
    qa_rows.append(dict(annotator=person,file=p.name,n_rows=len(d),
        unique_ids=d.verify_id.nunique(),duplicate_ids=int(d.verify_id.duplicated().sum()),
        invalid_labels=int(d.label.isna().sum()),circles=','.join(map(str,sorted(d.circle_id.dropna().unique()))),
        complete_flag_values=str(sorted(d.annotation_complete.dropna().astype(str).unique().tolist()))
             if 'annotation_complete' in d else '(absent)'))
    if d.verify_id.duplicated().any():raise ValueError(f'Duplicate verify_id in {p.name}')
    if d.label.isna().any():raise ValueError(f'Unexpected classification labels in {p.name}')
    raw_tables[person]=d
    table_rows.append(d[['verify_id','circle_id','label']].rename(columns={'circle_id':f'circle_{person}',
                                                                          'label':person}))
QA=pd.DataFrame(qa_rows).sort_values('annotator')
display(QA)
QA.to_csv(OUT/'NBAA_01_classification_file_audit.csv',index=False)
sets={who(p):set(raw_tables[who(p)].verify_id) for p in CLASS_CSV}
all_ids=set.union(*sets.values());common=set.intersection(*sets.values())
print('Union of IDs:',len(all_ids),'IDs present for every annotator:',len(common))
if len(common)!=len(all_ids):
    print('Missing IDs by annotator:',{name:sorted(all_ids-ids) for name,ids in sets.items()})
    print('Only complete matched IDs will enter five-expert agreement summaries.')
all_table=pd.DataFrame({'verify_id':sorted(all_ids)})
for t in table_rows:all_table=all_table.merge(t,on='verify_id',how='left',validate='one_to_one')
for person in raw_tables:
    derived=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
    bad=all_table[f'circle_{person}'].notna()&(derived!=all_table[f'circle_{person}'].astype('Int64'))
    assert not bad.any(),f'circle_id vs verify_id disagreement for {person}'
all_table['circle_id']=all_table.verify_id.str.extract(r'^C(\d+)_V\d+$')[0].astype('Int64')
EXPERTS=['Nadine','Thomas','Joseph','Marijn','Kristof']
assert set(EXPERTS)==set(raw_tables)
matched=all_table.dropna(subset=EXPERTS).copy()
matched[['verify_id','circle_id',*EXPERTS]].to_csv(OUT/'NBAA_01_aligned_labels.csv',index=False)
display(matched.groupby('circle_id').size().rename('candidate_crowns').reset_index())
display(pd.DataFrame({person:matched[person].value_counts() for person in EXPERTS}).fillna(0).astype(int))
print('No label harmonization beyond the explicit M/N/U code mapping.')

## 02 — Classification consistency (descriptive, not annotator 'accuracy')

In [ ]:
stats=[]
for a,b in itertools.combinations(EXPERTS,2):
    q=matched[[a,b]].dropna();definite=q[(q[a]!='U')&(q[b]!='U')]
    stats.append(dict(annotator_a=a,annotator_b=b,n=len(q),
        agreement=float((q[a]==q[b]).mean()),
        cohen_kappa=float(cohen_kappa_score(q[a],q[b],labels=['M','N','U'])),
        n_definite=len(definite),
        definite_agreement=float((definite[a]==definite[b]).mean()) if len(definite) else np.nan,
        definite_kappa=float(cohen_kappa_score(definite[a],definite[b],labels=['M','N']))
           if len(definite) and definite[a].nunique()>1 and definite[b].nunique()>1 else np.nan))
PAIR=pd.DataFrame(stats)
PAIR.to_csv(OUT/'NBAA_02_pairwise_classifications.csv',index=False)
display(PAIR.round(3))
votes=matched[EXPERTS].apply(lambda r:r.value_counts(),axis=1).fillna(0).astype(int)
for k in ['M','N','U']:
    matched[f'votes_{k}']=votes[k].values if k in votes else 0
matched['largest_agreement']=matched[[f'votes_{x}' for x in 'MNU']].max(axis=1)
matched['majority_label']=matched.apply(lambda r: next((k for k in 'MNU' if r[f'votes_{k}']>=3),'none'),axis=1)
display(matched.groupby(['circle_id','largest_agreement']).size().unstack(fill_value=0))
display(matched.groupby('majority_label').size().rename('crowns'))
matched.to_csv(OUT/'NBAA_02_crown_vote_table.csv',index=False)
print('NOTE: agreement measures label similarity, NOT botanical species correctness.')

## 03 — Polygon geometry audit and in-memory topology repair

Some digitized polygons are self-intersecting or topologically invalid. Preserve original geometry quality in a CSV and use `make_valid` on an in-memory analytical copy before unions and IoU. Geometry changes are quantified; source files are never modified.

In [ ]:
POLY_FILES=sorted(p for p in SRC.iterdir() if p.name.startswith('Musanga_Polygones_'))
print('Polygon export files:',[p.name for p in POLY_FILES])
assert len(POLY_FILES)==4, f'Expected 4 polygon exports; found {len(POLY_FILES)}'
from shapely import make_valid
from shapely.geometry import MultiPolygon, Polygon, GeometryCollection
from shapely.ops import unary_union

def polygonal_parts(geom):
    """Preserve only actual polygon components after GEOS repair."""
    if geom is None or geom.is_empty:return []
    if isinstance(geom,Polygon):return [geom]
    if isinstance(geom,MultiPolygon):return list(geom.geoms)
    if hasattr(geom,'geoms'):
        return [p for g in geom.geoms for p in polygonal_parts(g)]
    return []

def repair_polygon_for_analysis(geom):
    """Only an in-memory analysis copy; never overwrite or silently fix source."""
    if geom is None or geom.is_empty:return None
    fixed=make_valid(geom) if not geom.is_valid else geom
    polys=[p for p in polygonal_parts(fixed) if p.is_valid and p.area>0]
    if not polys:return None
    result=unary_union(polys)
    return result if not result.is_empty else None

def safe_union(geometries):
    cleaned=[g for g in geometries if g is not None and not g.is_empty]
    if not cleaned:return GeometryCollection()
    return unary_union(cleaned)

POLY={}
poly_quality=[]
GEOM_QA=[]
UTM='EPSG:32635'
for p in POLY_FILES:
    person=who(p)
    raw=json.loads(p.read_text(encoding='utf-8-sig'))
    assert raw.get('type')=='FeatureCollection' and isinstance(raw.get('features'),list),p.name
    entries=[]
    for i,f in enumerate(raw['features']):
        props=f.get('properties') or {}
        geom=shape(f['geometry']) if f.get('geometry') else None
        entries.append({**props,'geometry':geom,'source_row':i})
    orig=gpd.GeoDataFrame(entries,geometry='geometry',crs='EPSG:4326')
    assert 'circle_id' in orig and 'crown_id' in orig,p.name
    orig['circle_id']=pd.to_numeric(orig.circle_id,errors='raise').astype(int)
    # Work in metric CRS. Keep the raw geometry independently for audit.
    orig=orig.to_crs(UTM)
    clean_geoms=[]
    for _,r in orig.iterrows():
        geom=r.geometry
        was_valid=(geom is not None and not geom.is_empty and geom.is_valid)
        original_area=float(geom.area) if geom is not None and not geom.is_empty else 0.
        fixed=repair_polygon_for_analysis(geom)
        repaired_area=float(fixed.area) if fixed is not None else np.nan
        GEOM_QA.append(dict(annotator=person,source_row=int(r.source_row),
             circle_id=int(r.circle_id),crown_id=str(r.crown_id),
             original_valid=bool(was_valid),original_geom_type=geom.geom_type if geom is not None else '',
             original_area_m2=original_area,repaired_area_m2=repaired_area,
             area_change_pct=100*(repaired_area-original_area)/original_area if original_area>0 else np.nan,
             geometry_usable=bool(fixed is not None),
             repaired=bool(not was_valid and fixed is not None)))
        clean_geoms.append(fixed)
    # All downstream area/IoU/matching/visualization operations use validated copies.
    d=orig.copy()
    d['geometry']=gpd.GeoSeries(clean_geoms,index=d.index,crs=UTM)
    if d.geometry.isna().any():
        print('WARNING: unusable polygon geometry excluded from spatial comparisons:',
              person,d.loc[d.geometry.isna(),['circle_id','crown_id','source_row']].to_dict('records'))
        d=d.loc[d.geometry.notna()].copy()
    assert d.geometry.is_valid.all(),f'Invalid geometry remains for {person}'
    d['area_m2']=d.geometry.area
    d['centroid_x']=d.geometry.centroid.x
    d['centroid_y']=d.geometry.centroid.y
    d['annotator']=person
    POLY[person]=d
    for cid,g in d.groupby('circle_id'):
        geom_union=safe_union(g.geometry.tolist())
        rows_for_circle=[x for x in GEOM_QA if x['annotator']==person and x['circle_id']==int(cid)]
        poly_quality.append(dict(annotator=person,circle_id=int(cid),n_polygons=len(g),
             sum_polygon_area_m2=float(g.area_m2.sum()),
             union_area_m2=float(geom_union.area),
             median_polygon_area_m2=float(g.area_m2.median()),
             duplicate_crown_ids=int(g.crown_id.duplicated().sum()),
             n_invalid_original=sum(not x['original_valid'] for x in rows_for_circle),
             n_repaired=sum(x['repaired'] for x in rows_for_circle),
             n_unusable=sum(not x['geometry_usable'] for x in rows_for_circle),
             annotation_status=str(g.annotation_status.value_counts().to_dict())
                 if 'annotation_status' in g else '(absent)'))
GEOM_AUDIT=pd.DataFrame(GEOM_QA)
GEOM_AUDIT.to_csv(OUT/'NBAA_03_geometry_repair_audit.csv',index=False)
POLY_QA=pd.DataFrame(poly_quality).sort_values(['circle_id','annotator'])
POLY_QA.to_csv(OUT/'NBAA_03_polygon_coverage_by_circle.csv',index=False)
display(POLY_QA.round(2))
print('=== Original geometry validity and in-memory repairs ===')
display(GEOM_AUDIT.groupby('annotator').agg(
    original_invalid=('original_valid',lambda x:int((~x).sum())),
    repaired=('repaired','sum'),unusable=('geometry_usable',lambda x:int((~x).sum())),
    max_abs_area_change_pct=('area_change_pct',lambda x:float(x.abs().max()))).round(3))
matrix=POLY_QA.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int)
display(matrix)
print('Usable polygon counts:',{a:len(d) for a,d in POLY.items()})
print('All original source files are unchanged. Subsequent sections use repaired analytical copies.')
print('Crucial: missing polygon != non-Musanga classification; completeness must be established.')


## 04 — Pairwise *spatial* agreement where both annotators drew polygons

In [ ]:
# Area-level IoU on comparable circle-specific polygon unions.
# Detection overlap alone cannot distinguish two species without independent truth.
pair_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    overlap_circles=sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id))
    for cid in overlap_circles:
        aa=POLY[a].loc[POLY[a].circle_id==cid]
        bb=POLY[b].loc[POLY[b].circle_id==cid]
        ga=unary_union(aa.geometry.tolist());gb=unary_union(bb.geometry.tolist())
        inter=ga.intersection(gb).area;uni=ga.union(gb).area
        pair_rows.append(dict(circle_id=cid,annotator_a=a,annotator_b=b,
          n_a=len(aa),n_b=len(bb),area_a_m2=float(ga.area),area_b_m2=float(gb.area),
          intersection_m2=float(inter),union_m2=float(uni),
          area_iou=float(inter/uni) if uni>0 else np.nan,
          a_covered_by_b=float(inter/ga.area) if ga.area>0 else np.nan,
          b_covered_by_a=float(inter/gb.area) if gb.area>0 else np.nan))
SPATIAL=pd.DataFrame(pair_rows)
SPATIAL.to_csv(OUT/'NBAA_04_pairwise_polygon_union_IoU.csv',index=False)
display(SPATIAL.sort_values(['circle_id','annotator_a','annotator_b']).round(3))
display(SPATIAL.groupby(['annotator_a','annotator_b']).agg(
  circles=('circle_id','count'),mean_area_IoU=('area_iou','mean'),
  median_area_IoU=('area_iou','median')).round(3))
print('These are unsigned overlap measures; polygons can represent different crowns or delineation rules.')

## 05 — Crown-to-crown spatial matching (exploratory, threshold sensitivity)

In [ ]:
def match_iou(A,B,minimum):
    # IoU is computed in projected metres; optimal one-to-one Hungarian assignment.
    if not len(A) or not len(B):return dict(matches=0,iou_mean=np.nan,frac_a=0,frac_b=0)
    mat=np.zeros((len(A),len(B)),dtype=float)
    for i,ga in enumerate(A.geometry):
        for j,gb in enumerate(B.geometry):
            if not ga.intersects(gb):continue
            u=ga.union(gb).area
            mat[i,j]=ga.intersection(gb).area/u if u>0 else 0
    rr,cc=linear_sum_assignment(-mat)
    good=mat[rr,cc]>=minimum
    return dict(matches=int(good.sum()),
        iou_mean=float(mat[rr[good],cc[good]].mean()) if good.any() else np.nan,
        frac_a=float(good.sum()/len(A)),frac_b=float(good.sum()/len(B)))
match_rows=[]
for a,b in itertools.combinations(POLY.keys(),2):
    for cid in sorted(set(POLY[a].circle_id)&set(POLY[b].circle_id)):
        da=POLY[a].loc[POLY[a].circle_id==cid]
        db=POLY[b].loc[POLY[b].circle_id==cid]
        for threshold in (.10,.30,.50):
            match_rows.append(dict(circle_id=cid,a=a,b=b,iou_threshold=threshold,
                n_a=len(da),n_b=len(db),**match_iou(da,db,threshold)))
MATCH=pd.DataFrame(match_rows)
MATCH.to_csv(OUT/'NBAA_05_one_to_one_crown_matches.csv',index=False)
display(MATCH.groupby(['a','b','iou_threshold']).agg(
    circles=('circle_id','count'),matched_crowns=('matches','sum'),
    avg_fraction_a=('frac_a','mean'),avg_fraction_b=('frac_b','mean')).round(3))
print('Unmatched crown need not be a false positive: reference completeness is unverified.')

## 06 — Compact site map / visual QA and exploration hand-off

In [ ]:
# Small vector-only visualization: no large TIFF reads.
from matplotlib.patches import Patch
colors={'Nadine':'#1e7962','Thomas':'#cf7c37','Joseph':'#6178b8','Marijn':'#ad589c'}
circles=sorted(set.union(*(set(d.circle_id) for d in POLY.values())))
fig,axes=plt.subplots(2,3,figsize=(15,10))
for ax,cid in zip(axes.flat,circles):
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        d=POLY.get(person)
        if d is None:continue
        subset=d[d.circle_id==cid]
        if len(subset):subset.boundary.plot(ax=ax,color=colors[person],linewidth=1,alpha=.8)
    ax.set_title(f'Circle {cid} — independently digitized crowns')
    ax.set_aspect('equal');ax.tick_params(labelsize=7)
for ax in axes.flat[len(circles):]:ax.axis('off')
fig.legend(handles=[Patch(facecolor='none',edgecolor=colors[n],label=n) for n in colors],
           loc='lower center',ncol=4)
plt.tight_layout(rect=[0,.04,1,1])
fig.savefig(OUT/'NBAA_06_polygon_overlay_by_circle.png',dpi=170)
plt.show();plt.close(fig)
contract=dict(schema='NBAA_v4_exploratory_1',
  source_directory=str(SRC),output_directory=str(OUT),
  classification_files={p:raw_tables[p].shape[0] for p in EXPERTS},
  common_classification_crowns=int(len(matched)),
  digitized_polygon_counts={p:int(len(POLY[p])) for p in POLY},
  classification_policy='Uncertain is a separate observed label; missing is not negative',
  polygon_policy='Independent positive polygons; absent polygons not negative; no automatic truth',
  alignment='CSV verify_id joins; polygon overlays by projected geometry and circle',
  train_test_policy='No training or new splits in this audit',
  further_review=['Digitizing completeness for each circle and annotator',
                  'Compare classification vs polygon tasks only where identity/coverage established',
                  'Independent RGB/DSM embeddings and spatially held-out annotator predictors',
                  'Repeat blinded annotations to assess genuine intra-annotator repeatability'])
(OUT/'NBAA_manifest.json').write_text(json.dumps(contract,indent=2))
print('NBAA exploration complete. Outputs:')
for p in sorted(OUT.glob('NBAA_*')):print(' ',p.name)
print('NEXT DECISION: review counts, completeness, overlap and image QA before modelling.')

## 07 — Which annotators outlined the **same crown**? (exploratory spatial object agreement)

**Critical distinction:** a zero in the polygon count table means **no polygon record**, *not* an observed zero-Musanga survey. Every circle without polygons is marked `not_recorded`; circles with polygons are marked `polygons_present_completion_unverified`, because all source polygon records have `annotation_status='draft'`. Only independently confirmed complete surveys can support false-negative/detection-rate interpretation.

This block identifies spatially overlapping polygon groups. An edge requires **different annotators**, geometrical intersection, polygon IoU ≥0.10 *or* at least 35% of the smaller polygon covered, plus nearby centroids. Connected groups with **multiple polygons from the same annotator** are flagged `ambiguous_group` and *not included in reliable 1/2/3/4 agreement counts*. Results are sensitive to these explicitly shown thresholds. Spatial overlap is not a species-truth vote.

Outputs: per-circle coverage, per-crown member table, group-level number of annotators (1–4), counts by circle, diagnostics/ambiguity, and plots. Visual overlay displays are vector-based; a georeferenced UAV RGB clip is added only if found locally and raster windows can be read safely.

In [ ]:
# NBAA Section 07A — coverage matrix and conservative spatial crown-group graph.
# Only reuses already-loaded POLY (repaired, independent) and reads no mosaics.
from collections import defaultdict
from itertools import combinations
from scipy.spatial.distance import euclidean
from shapely.geometry import box
import networkx as nx
try:
    import rasterio
    from rasterio.windows import from_bounds as raster_window_from_bounds
except ImportError:
    rasterio=None

annotators=['Nadine','Thomas','Joseph','Marijn']
circles=sorted(set().union(*[set(df.circle_id.unique()) for df in POLY.values()]))
coverage=[]
for circle in circles:
    for person in annotators:
        df=POLY[person];part=df[df.circle_id==circle]
        coverage.append(dict(circle_id=int(circle),annotator=person,
          n_polygons=int(len(part)),
          status='polygons_present_completion_unverified' if len(part) else 'not_recorded',
          complete_verified=False))
COVERAGE=pd.DataFrame(coverage)
COVERAGE.to_csv(OUT/'NBAA_07_circle_annotator_coverage.csv',index=False)
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='status'))
display(COVERAGE.pivot(index='circle_id',columns='annotator',values='n_polygons').fillna(0).astype(int))
print('CAUTION: zero records are unassessed, not confirmed zero Musanga.')
print('Even where polygons exist, full-circle completeness is not independently established.')

MIN_IOU=.10
MIN_SMALLER_OVERLAP=.35
MAX_CENTROID_FACTOR=1.5 # times the sum of equivalent-area circle radii
members=[];groups=[];edges=[]
for cid in circles:
    records=[]
    for who_ in annotators:
        d=POLY[who_]
        for _,r in d[d.circle_id==cid].iterrows():
            records.append(dict(annotator=who_,crown_id=str(r.crown_id),
                source_row=int(r.source_row),geometry=r.geometry,
                area_m2=float(r.geometry.area)))
    graph=nx.Graph();graph.add_nodes_from(range(len(records)))
    for i,j in combinations(range(len(records)),2):
        a,b=records[i],records[j]
        if a['annotator']==b['annotator']:continue
        ga,gb=a['geometry'],b['geometry']
        if not ga.intersects(gb):continue
        inter=ga.intersection(gb).area
        if inter<=0:continue
        union=ga.area+gb.area-inter
        iou=inter/union if union else 0
        smaller=inter/min(ga.area,gb.area) if min(ga.area,gb.area)>0 else 0
        ra=np.sqrt(ga.area/np.pi);rb=np.sqrt(gb.area/np.pi)
        dist=ga.centroid.distance(gb.centroid)
        if (iou>=MIN_IOU or smaller>=MIN_SMALLER_OVERLAP) and dist<=MAX_CENTROID_FACTOR*(ra+rb):
            graph.add_edge(i,j,weight=float(iou))
            edges.append(dict(circle_id=int(cid),annotator_a=a['annotator'],annotator_b=b['annotator'],
              crown_a=a['crown_id'],crown_b=b['crown_id'],
              iou=float(iou),smaller_overlap=float(smaller),centroid_distance_m=float(dist)))
    for group_idx,comp in enumerate(sorted(nx.connected_components(graph),key=lambda s:min(s)),start=1):
        rows=[records[i] for i in sorted(comp)]
        unique_people={r['annotator'] for r in rows}
        collisions=[person for person in unique_people if sum(r['annotator']==person for r in rows)>1]
        ambiguous=bool(collisions)
        gid=f'C{cid:02d}_G{group_idx:03d}'
        geoms=[r['geometry'] for r in rows]
        union=safe_union(geoms)
        groups.append(dict(group_id=gid,circle_id=int(cid),
            n_polygons=len(rows),n_annotators=len(unique_people),
            annotators=';'.join(sorted(unique_people)),
            ambiguous_group=ambiguous,
            same_annotator_collisions=';'.join(sorted(collisions)),
            member_ids=';'.join(f"{r['annotator']}:{r['crown_id']}" for r in rows),
            centroid_easting=float(union.centroid.x),
            centroid_northing=float(union.centroid.y),
            union_area_m2=float(union.area)))
        for r in rows:
            members.append(dict(group_id=gid,circle_id=int(cid),
                annotator=r['annotator'],crown_id=r['crown_id'],
                source_row=r['source_row'],polygon_area_m2=r['area_m2'],
                ambiguous_group=ambiguous))
GROUPS=pd.DataFrame(groups)
MEMBERS=pd.DataFrame(members)
EDGES=pd.DataFrame(edges)
for name,frame in [('NBAA_07_crown_groups.csv',GROUPS),
                    ('NBAA_07_crown_members.csv',MEMBERS),
                    ('NBAA_07_spatial_match_edges.csv',EDGES)]:
    frame.to_csv(OUT/name,index=False)
print('Spatial crown groups (not verified true crowns):',len(GROUPS))
print('Ambiguous chained groups:',int(GROUPS.ambiguous_group.sum()))
display(GROUPS.groupby(['n_annotators','ambiguous_group']).size().rename('groups').reset_index())
reliable=GROUPS.loc[~GROUPS.ambiguous_group]
print('Crown groups by number of annotators, excluding ambiguous links:')
display(reliable.groupby(['circle_id','n_annotators']).size().unstack(fill_value=0).reindex(columns=[1,2,3,4],fill_value=0))
print('Full group identity table (inspect before interpreting consensus):')
display(GROUPS[['group_id','circle_id','n_polygons','n_annotators','annotators',
                'ambiguous_group','member_ids']].head(35))


## 08 — High-resolution RGB inspection on original V3_1 UAV mosaics

Uses the **authoritative original RGB paths** from the frozen V3_1 NB00 audit manifest, not pre-cut verification clips. For each circle, it selects the mosaic by georeferenced intersection, reads a window at **native UAV pixel resolution**, overlays the four annotators' crown boundaries, and records the actual source resolution and any required downsampling. No whole mosaics are loaded. The large PNG figures are written to `08_high_resolution_rgb/`.

**Note:** the overlays remain visual QA, not proven correct crown identity. Source TIFFs stay read-only.

In [ ]:
# NBAA Section 08 — original HIGH-RES UAV RGB mosaics from frozen V3_1 NB00 manifest.
# No pre-cut clips are required. Only small exact native-resolution windows are read.
import json, math
import numpy as np, pandas as pd, geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
from rasterio.windows import from_bounds as window_from_bounds, Window, transform as window_transform
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds
from shapely.geometry import box
from matplotlib.patches import Patch
import matplotlib.patheffects as pe

# Authoritative audited RGB source manifest shared by V3_1 and V4 (do not infer paths).
manifest=ROOT/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert manifest.is_file(),('V3_1 NB00 manifest missing:',str(manifest))
audit=json.loads(manifest.read_text())
assert audit.get('schema_version')=='v3_1_input_audit_kvo_final',audit.get('schema_version')
RGB_SOURCES={r['file']:Path(r['path']) for r in audit['mosaics']
             if r['file']!='Mos_3_4_transparent_mosaic_group1_georef.tif'}
assert len(RGB_SOURCES)==5 and all(p.is_file() for p in RGB_SOURCES.values()),RGB_SOURCES
print('Using original five audited RGB UAV mosaics:')
for k,p in RGB_SOURCES.items(): print(' ',k,'→',p)

COLORS={'Nadine':'#00ff56','Thomas':'#ffff00','Joseph':'#00eaff','Marijn':'#ff00f5'}
SHORT={'Nadine':'Nad','Thomas':'Thom','Joseph':'Jo','Marijn':'Ma'}
CIRCLE_BOUNDARY_FILE=Path(audit['exhaustive_annotations']['complete_roi_boundaries'])
assert CIRCLE_BOUNDARY_FILE.is_file(),CIRCLE_BOUNDARY_FILE
SURVEY_CIRCLES=gpd.read_file(CIRCLE_BOUNDARY_FILE).reset_index(drop=True)
# V3_1 original contract: circle_id is the ORIGINAL zero-based shapefile row,
# not a pre-existing attribute (see V3_1 NB01 section 04).
assert len(SURVEY_CIRCLES)>=34,len(SURVEY_CIRCLES)
SURVEY_CIRCLES['circle_id']=np.arange(len(SURVEY_CIRCLES),dtype=int)
RGB_OUTPUT=OUT/'08_high_resolution_rgb'
RGB_OUTPUT.mkdir(parents=True,exist_ok=True)
# Native resolution is always retained unless an especially large output
# requires an explicit cap (logged in QA), not silently downsampled.
MAX_OVERVIEW_PX=8500
PAD_M=8.0
overview_rows=[]

def rgb_uint8(x, ds):
    """Convert 3 RGB bands into display RGB; preserve uint8, rescale other types for viewing."""
    if x.dtype==np.uint8:return np.moveaxis(x,0,-1)
    y=x.astype('float32')
    # 16-bit images may use 0..65535; use metadata-independent robust display stretch.
    out=np.empty_like(y,dtype=np.uint8)
    for k in range(min(3,y.shape[0])):
        v=y[k];valid=np.isfinite(v)
        lo,hi=np.nanpercentile(v[valid],[1,99]) if valid.any() else (0,1)
        out[k]=np.uint8(np.clip((v-lo)/(hi-lo if hi>lo else 1)*255,0,255))
    return np.moveaxis(out,0,-1)

def native_window(ds, bounds, pad_m=0):
    l,b,r,t=bounds
    w=window_from_bounds(l-pad_m,b-pad_m,r+pad_m,t+pad_m,transform=ds.transform)
    # Clip to actual image: a circle at a tile edge is never read out-of-bounds.
    w=w.round_offsets().round_lengths()
    return w.intersection(Window(0,0,ds.width,ds.height))

for cid in circles:
    # Each group is already a repaired polygon in UTM35N; find the SOURCE RGB
    # mosaic covering most of the group's extent (works for all four forest sites).
    parts={a:POLY[a].loc[POLY[a].circle_id==cid] for a in COLORS}
    all_geoms=[g for p in parts.values() for g in p.geometry]
    if not all_geoms:continue
    target=safe_union(all_geoms)
    candidates=[]
    for name,path in RGB_SOURCES.items():
        with rasterio.open(path) as ds:
            if ds.crs is None:continue
            xmin,ymin,xmax,ymax=transform_bounds(ds.crs,'EPSG:32635',*ds.bounds)
            frac=target.intersection(box(xmin,ymin,xmax,ymax)).area/target.area
            candidates.append((frac,name,path))
    frac,name,path=max(candidates,key=lambda x:x[0])
    if frac<.90: print(f'WARNING: circle {cid}: best RGB footprint intersection {frac:.1%}; inspect alignment.')
    with rasterio.open(path) as ds:
        if ds.crs is None or ds.count<3:raise ValueError(f'Unexpected RGB source {path}')
        projected={a:p.to_crs(ds.crs) for a,p in parts.items()}
        geometry=gpd.GeoSeries([target],crs='EPSG:32635').to_crs(ds.crs).iloc[0]
        survey=SURVEY_CIRCLES.loc[SURVEY_CIRCLES.circle_id==cid].to_crs(ds.crs)
        assert len(survey)==1, f'Circle boundary {cid} missing'
        circle_geom=survey.geometry.iloc[0]
        win=native_window(ds,circle_geom.bounds,pad_m=PAD_M)
        native_w,native_h=int(win.width),int(win.height)
        factor=max(1,math.ceil(max(native_w,native_h)/MAX_OVERVIEW_PX))
        out_h=max(1,math.ceil(native_h/factor));out_w=max(1,math.ceil(native_w/factor))
        raw=ds.read([1,2,3],window=win,out_shape=(3,out_h,out_w),
                    resampling=Resampling.average if factor>1 else Resampling.nearest)
        arr=rgb_uint8(raw,ds)
        actual_bounds=rasterio.windows.bounds(win,ds.transform)
        left,bottom,right,top=actual_bounds
        fig,ax=plt.subplots(figsize=(14,14*arr.shape[0]/arr.shape[1]),dpi=130)
        ax.imshow(arr,extent=(left,right,bottom,top),interpolation='nearest')
        for a,pg in projected.items():
            if len(pg):
                pg.plot(ax=ax,facecolor=COLORS[a],edgecolor='none',alpha=.09)
                pg.boundary.plot(ax=ax,edgecolor='black',linewidth=3.6,alpha=.95)
                pg.boundary.plot(ax=ax,edgecolor=COLORS[a],linewidth=2.3,alpha=1)
                for pg_geom in pg.geometry:
                    pt=pg_geom.representative_point()
                    ax.text(pt.x,pt.y,SHORT[a],color=COLORS[a],fontsize=9,
                            fontweight='bold',ha='center',va='center',zorder=15,
                            path_effects=[pe.Stroke(linewidth=2.5,foreground='black'),pe.Normal()])
        gpd.GeoSeries([circle_geom],crs=ds.crs).boundary.plot(ax=ax,color='black',linewidth=5,linestyle='--',zorder=18)
        gpd.GeoSeries([circle_geom],crs=ds.crs).boundary.plot(ax=ax,color='white',linewidth=3,linestyle='--',zorder=19)
        for _,r in GROUPS[GROUPS.circle_id==cid].iterrows():
            ctr=gpd.GeoSeries([gpd.points_from_xy([r.centroid_easting],[r.centroid_northing])[0]],
                    crs='EPSG:32635').to_crs(ds.crs).iloc[0]
            ax.text(ctr.x,ctr.y,r.group_id.split('_')[-1]+('!' if r.ambiguous_group else ''),
                color='white',fontsize=7,ha='center',va='center',
                bbox=dict(facecolor='black',alpha=.60,pad=1.0,edgecolor='none'))
        ax.set(xlim=(left,right),ylim=(bottom,top),
              title=f'Circle {cid} | original RGB: {name} | source {native_w}×{native_h}px | read factor {factor}')
        ax.set_aspect('equal')
        fig.legend(handles=[Patch(facecolor=COLORS[a],edgecolor='black',label=f'{SHORT[a]} = {a}') for a in COLORS],
            loc='lower center',ncol=4)
        fig.tight_layout(rect=[0,.025,1,1])
        dest=RGB_OUTPUT/f'circle_{cid:02d}_RGB_polygon_overlay.png'
        fig.savefig(dest,dpi=200)
        plt.show();plt.close(fig)
        overview_rows.append(dict(circle_id=int(cid),source=name,
              source_path=str(path),source_resolution_m=float(abs(ds.transform.a)),
              rgb_coverage_fraction=float(frac),native_window_width_px=native_w,
              native_window_height_px=native_h,read_downsample_factor=factor,
              rendered_width_px=out_w,rendered_height_px=out_h,figure=str(dest)))
OVERVIEW_QA=pd.DataFrame(overview_rows)
OVERVIEW_QA.to_csv(OUT/'NBAA_08_rgb_source_and_resolution_audit.csv',index=False)
display(OVERVIEW_QA[['circle_id','source','source_resolution_m','rgb_coverage_fraction',
                     'native_window_width_px','native_window_height_px','read_downsample_factor']])
print('Full-resolution RGB polygon overlays:',RGB_OUTPUT)
print('Each overlay uses the source RGB raster window; no full mosaic is loaded.')
print('For crown-edge evaluation, use native-resolution source windows rather than stretching small clips.')


## 09 — Cross-task consistency: 55 labelled crowns versus independent digitization

**Source fix:** the original `Verification.shp` does not contain the canonical `verify_id` schema assumed in the first version. This section now reads the **verified GEE canonical** `verification_unique.geojson` (from `Final/Outputs/V3/06_GEE_Verification_v2`) with its explicit `verify_id` and `circle_id` fields. If that file is unavailable, it uses `verification_68_source_mapping.csv` to reconcile raw shapefile rows to canonical crown IDs; it will stop rather than guess. The source has 60 unique candidate polygons; this analysis uses exactly the **55 classified by all five experts**.

Compare class votes against independent positive polygon digitization only where a matched crown is found. Missing survey or no polygon does **not** imply a negative identification. Exact matching is audited, and V3_1/V4 training data are not modified.

In [ ]:
# NBAA Section 09 — conservative candidate polygon versus manually drawn crown.
# The working GEE verification data are keyed by verify_id; the original
# Verification.shp is a raw digitization file and does NOT guarantee this schema.
# Prefer the validated canonical 60-feature GeoJSON produced by GEE Verification v2.
VERIFY_GEOJSON_OVERRIDE=None # optional exact path, if data were moved
VERIFY_DIR_CANDIDATES=[
    ROOT/'Outputs/V3/06_GEE_Verification_v2',   # ACTUAL connected Drive folder
    ROOT/'Outputs/V3_1/06_GEE_Verification_v2',
    ROOT/'Outputs/v3/06_GEE_Verification_v2',
    ROOT/'Outputs/06_GEE_Verification_v2',
    SRC]
paths=([Path(VERIFY_GEOJSON_OVERRIDE)] if VERIFY_GEOJSON_OVERRIDE else [])+[
    folder/'verification_unique.geojson' for folder in VERIFY_DIR_CANDIDATES]
verified_geojson=next((p for p in paths if p.is_file()),None)
if verified_geojson:
    print('Using verified GEE canonical geometry:',verified_geojson)
    src_verify=gpd.read_file(verified_geojson)
else:
    # Fall back to the original raw shapefile, reconciled using the authoritative
    # GEE source-row -> canonical verify_id mapping from the same V3 output folder.
    mapping_paths=[folder/'verification_68_source_mapping.csv' for folder in VERIFY_DIR_CANDIDATES]
    mapping_file=next((p for p in mapping_paths if p.is_file()),None)
    assert mapping_file is not None,(
        'Cannot identify crown geometry by verify_id. Original Verification.shp '
        'does not provide canonical ids. Expected verified_geojson or GEE '
        'verification_68_source_mapping.csv in Outputs/V3/06_GEE_Verification_v2. '
        'Set VERIFY_GEOJSON_OVERRIDE to the exact verified GeoJSON path.')
    VERIFY=SRC/'Verification.shp'
    assert VERIFY.is_file(),VERIFY
    source=gpd.read_file(VERIFY).reset_index(drop=True)
    source['source_row']=np.arange(len(source))
    mapping=pd.read_csv(mapping_file)
    assert {'source_row','canonical_id','is_duplicate'}.issubset(mapping.columns)
    assert mapping.source_row.is_unique
    assert len(mapping)==len(source),(len(mapping),len(source))
    canonical=mapping.loc[~mapping.is_duplicate.astype(str).str.lower().eq('true'),
                          ['source_row','canonical_id','circle_id']].copy()
    src_verify=source.merge(canonical,on='source_row',how='inner',validate='one_to_one')
    src_verify=src_verify.rename(columns={'canonical_id':'verify_id'})
    print('Using raw shapefile reconciled to GEE canonical ids via:',mapping_file)
print('Geometry source columns:',src_verify.columns.tolist())
assert {'verify_id','circle_id'}.issubset(src_verify.columns),src_verify.columns.tolist()
src_verify['verify_id']=src_verify.verify_id.astype(str).str.strip()
print('Reference source raw:',len(src_verify),'rows /',src_verify.verify_id.nunique(),'unique IDs')
sub=src_verify.loc[src_verify.verify_id.isin(matched.verify_id)].to_crs(UTM).copy()
assert set(sub.verify_id)==set(matched.verify_id),{
  'missing':sorted(set(matched.verify_id)-set(sub.verify_id)),
  'extra':sorted(set(sub.verify_id)-set(matched.verify_id))}
if sub.verify_id.duplicated().any():
    print('Duplicate reference IDs: verify geometry equivalence before deduplication')
    for vid,part in sub.groupby('verify_id'):
        if len(part)>1:
            gg=[repair_polygon_for_analysis(g) for g in part.geometry]
            if any(g is None or not gg[0].equals_exact(g,1e-3) for g in gg[1:]):
                raise ValueError('Conflicting duplicate verify_id geometry: '+vid)
    sub=sub.drop_duplicates('verify_id').copy()
sub['geometry']=[repair_polygon_for_analysis(g) for g in sub.geometry]
assert sub.geometry.notna().all() and len(sub)==55
# Discard any source-side duplicate circle_id column and retain original canonical
# verification circle id. The classification table has the same IDs.
sub['circle_id']=pd.to_numeric(sub.circle_id,errors='raise').astype(int)
REF=sub.merge(matched[['verify_id',*EXPERTS]],on='verify_id',validate='one_to_one')
assert (REF.circle_id==REF.verify_id.str.extract(r'^C(\d+)_V')[0].astype(int)).all()
print('Verified 55 unique reference crown polygons with exact expert-label joins.')
cross=[]
for _,r in REF.iterrows():
    for person in ['Nadine','Thomas','Joseph','Marijn']:
        candidates=POLY[person].loc[POLY[person].circle_id==int(r.circle_id)]
        best=(0.,0.,None)
        for _,q in candidates.iterrows():
            inter=r.geometry.intersection(q.geometry).area
            if inter<=0:continue
            uni=r.geometry.union(q.geometry).area
            iou=inter/uni if uni else 0.
            frac=inter/min(r.geometry.area,q.geometry.area)
            if iou>best[0]:best=(iou,frac,str(q.crown_id))
        match=(best[0]>=.10 or best[1]>=.35)
        status='not_recorded' if len(candidates)==0 else 'polygons_present_completion_unverified'
        label=r[person]
        interpretation=('not_assessed' if status=='not_recorded' else
           'possible_cross_task_conflict' if match and label=='N' else
           'affirmative_concordance' if match and label=='M' else
           'uncertain_and_delineated' if match and label=='U' else
           'unmatched_completeness_unknown')
        cross.append(dict(verify_id=r.verify_id,circle_id=int(r.circle_id),annotator=person,
            classification=label,survey_status=status,matched=bool(match),
            best_polygon_id=best[2],best_iou=float(best[0]),
            fraction_of_smaller_polygon=float(best[1]),interpretation=interpretation))
CROSS=pd.DataFrame(cross)
CROSS.to_csv(OUT/'NBAA_09_cross_task_crown_match.csv',index=False)
display(CROSS.groupby(['annotator','classification','interpretation']).size().rename('n').reset_index())
display(CROSS.groupby(['annotator','classification']).agg(
    n=('verify_id','count'),matched=('matched','sum'),
    median_iou=('best_iou','median')).round(3))
print('Only candidate conflicts are flagged. Absence of polygon is NOT a negative label.')

## 10 — Independent crown features: RGB and DSM (55 common reference objects)

All 55 candidate crown polygons are shared across experts; extract **one** feature vector per reference crown from the native-resolution original RGB, regardless of who selected it. Add area, compactness, RGB chromaticity, within-crown variation, and **SfM DSM canopy-surface heterogeneity** where a validated DSM mosaic exists.

**Controls:** local small windows only, avoid loading the mosaics; no labels in the feature matrix; do not use MS until its radiometry is resolved. Source RGB band intensities are not consistently calibrated across acquisitions. DSM measures are canopy-surface, not terrain-normalized tree height. This is exploratory, not species ground truth.

In [ ]:
# NBAA Section 10: crown features, windowed RGB + optional DSM on RGB grid.
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds as win_from_bounds,Window
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from shapely.geometry import mapping
DSM_REG=ROOT/'Outputs/v4/01_reference_data_no_MOS34/01_dsm_file_registry.json'
DSM_PATHS={}
if DSM_REG.is_file():
    DSM_PATHS={k:Path(v) for k,v in json.loads(DSM_REG.read_text()).get('rgb_to_dsm',{}).items()}
RGB_BY_CIRCLE=OVERVIEW_QA.set_index('circle_id')['source'].to_dict()
assert set(REF.circle_id.astype(int)).issubset(RGB_BY_CIRCLE)
feats=[]
for _,r in REF.iterrows():
    rgbname=RGB_BY_CIRCLE[int(r.circle_id)]
    with rasterio.open(RGB_SOURCES[rgbname]) as rgb:
        geom=gpd.GeoSeries([r.geometry],crs=UTM).to_crs(rgb.crs).iloc[0]
        w=win_from_bounds(*geom.bounds,transform=rgb.transform).round_offsets().round_lengths()
        w=w.intersection(Window(0,0,rgb.width,rgb.height))
        if w.width<2 or w.height<2:continue
        tr=rasterio.windows.transform(w,rgb.transform)
        a=rgb.read([1,2,3],window=w,masked=True).astype('float64')
        mask=geometry_mask([mapping(geom)],out_shape=(int(w.height),int(w.width)),
                           transform=tr,invert=True)
        mask &= (~np.ma.getmaskarray(a)).all(axis=0)
        if mask.sum()<20:continue
        v=a.data[:,mask]
        raw=np.median(v,axis=1)
        chrom=v/np.maximum(v.sum(axis=0,keepdims=True),1.)
        means=v.mean(axis=0)
        feature=dict(verify_id=str(r.verify_id),circle_id=int(r.circle_id),mosaic=rgbname,
            area_m2=float(geom.area),compactness=float(4*np.pi*geom.area/max(geom.length**2,1e-10)),
            n_rgb_pixels=int(mask.sum()),RGB_R=float(raw[0]),RGB_G=float(raw[1]),RGB_B=float(raw[2]),
            color_R=float(np.median(chrom[0])),color_G=float(np.median(chrom[1])),
            color_B=float(np.median(chrom[2])),
            brightness_CV=float(np.std(means)/max(np.mean(means),1.)),
            RGB_texture=float(np.std(means)),
            DSM_valid_fraction=np.nan,DSM_roughness_m=np.nan,DSM_range_p90_p10_m=np.nan)
        dp=DSM_PATHS.get(rgbname)
        if dp is not None and dp.exists():
            with rasterio.open(dp) as ds, WarpedVRT(ds,crs=rgb.crs,transform=rgb.transform,
                 width=rgb.width,height=rgb.height,resampling=Resampling.bilinear) as zvrt:
                z=zvrt.read(1,window=w,masked=True)
            zdata=np.asarray(z.data,dtype='float64')
            zvalid=(~np.ma.getmaskarray(z))&np.isfinite(zdata)&(~np.isclose(zdata,0))
            good=zvalid&mask
            feature['DSM_valid_fraction']=float(good.sum()/mask.sum())
            if good.sum()>=20:
                zv=zdata[good]
                feature['DSM_roughness_m']=float(np.std(zv))
                feature['DSM_range_p90_p10_m']=float(np.percentile(zv,90)-np.percentile(zv,10))
        feats.append(feature)
FEATURES=pd.DataFrame(feats)
assert len(FEATURES)>0
FEATURES.to_csv(OUT/'NBAA_10_reference_crown_features.csv',index=False)
missing=sorted(set(matched.verify_id)-set(FEATURES.verify_id))
print(f'Feature extraction: {len(FEATURES)} / {len(matched)} reference crowns.')
if missing: print('WARNING: missing RGB window features:',missing)
display(FEATURES.groupby('circle_id').agg(
    crowns=('verify_id','count'),median_area_m2=('area_m2','median'),
    median_rgb_pixels=('n_rgb_pixels','median'),
    n_dsm=('DSM_roughness_m',lambda x:int(x.notna().sum()))).round(2))
display(FEATURES.head().round(3))

## 11 — Unsupervised visual phenotype clustering and annotator enrichment

Cluster the **55 shared reference crowns**, using only RGB/shape/available DSM features—not human labels or which annotator drew a polygon. Then compare the *same clusters* against all five annotators' positive/negative/uncertain votes. Check whether clusters merely reflect the site/circle, RGB illumination, or crown size. Two to four cluster solutions are explored.

**Scientific caution:** These are image-feature groups, **not identified tree species**. With 55 crowns in six spatially clustered circles, cluster stability and significance are weak; use them to generate hypotheses for botanical field verification, not to assert Nadine mixed two species.

In [ ]:
# NBAA Section 11 — label-free cluster discovery; human-vote enrichment afterward.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
FEATURE_COLUMNS=['area_m2','compactness','color_R','color_G','color_B',
  'brightness_CV','RGB_texture','DSM_roughness_m','DSM_range_p90_p10_m']
d=FEATURES.merge(matched[['verify_id',*EXPERTS]],on='verify_id',validate='one_to_one')
available=[col for col in FEATURE_COLUMNS if d[col].notna().sum()>=max(12,int(.6*len(d)))]
assert len(available)>=5,('Insufficient crown features',available)
X=RobustScaler().fit_transform(SimpleImputer(strategy='median').fit_transform(
    d[available].replace([np.inf,-np.inf],np.nan)))
coords=PCA(n_components=2).fit_transform(X)
d['PC1'],d['PC2']=coords[:,0],coords[:,1]
solutions=[]
for k in (2,3,4):
    model=KMeans(n_clusters=k,random_state=2026,n_init=25).fit(X)
    solutions.append((k,float(silhouette_score(X,model.labels_)),model.labels_))
print('Unsupervised k sensitivity (k,silhouette):',[(k,round(sc,3)) for k,sc,_ in solutions])
k,sc,labels=max(solutions,key=lambda x:x[1])
d['cluster']=labels
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for cluster,part in d.groupby('cluster'):
    axes[0].scatter(part.PC1,part.PC2,label=f'cluster {cluster}, n={len(part)}',alpha=.85)
for circle,part in d.groupby('circle_id'):
    axes[1].scatter(part.PC1,part.PC2,label=f'circle {circle}',alpha=.85)
for ax in axes:
    ax.set(xlabel='PCA 1',ylabel='PCA 2')
    ax.legend(fontsize=8)
axes[0].set_title('Crown clusters (no human labels used)')
axes[1].set_title('Identical PCA, coloured by circle')
fig.savefig(OUT/'NBAA_11_clusters_vs_circle.png',dpi=180)
plt.show();plt.close(fig)
profiles=[]
for cl,part in d.groupby('cluster'):
    for person in EXPERTS:
        profiles.append(dict(cluster=int(cl),annotator=person,n=int(len(part)),
           n_M=int((part[person]=='M').sum()),n_N=int((part[person]=='N').sum()),
           n_U=int((part[person]=='U').sum()),
           proportion_M=float((part[person]=='M').mean())))
PROFILES=pd.DataFrame(profiles)
PROFILES.to_csv(OUT/'NBAA_11_cluster_by_annotator.csv',index=False)
d.to_csv(OUT/'NBAA_11_cluster_crown_assignments.csv',index=False)
display(PROFILES.pivot(index='cluster',columns='annotator',values='proportion_M').round(3))
print('Cluster composition by circle (important confounding test):')
display(pd.crosstab(d.cluster,d.circle_id))
print('Chosen exploratory k=',k,'silhouette=',round(sc,3))
print('Next: inspect matched RGB crown images by cluster before any species hypothesis.')

## 12 — Crown-level visual consistency (no U-Net)

We now use **all reference and independently digitized crown objects**, extract frozen ImageNet ResNet18 features from RGB imagery, and fit small per-annotator classifiers. No neural-network training, segmentation chips, epochs, or pixel-level threshold fitting.

The 55 explicit reference M/N/U votes are preserved; independent digitized polygons add positive labels. For circles with digitized polygons, assume exhaustive surveying (PI-confirmed objective): other candidate crown objects in that circle are provisional negatives unless the annotator explicitly classified them U or M. Unmapped circles are not assessed. Match duplicate crown objects spatially first; exclude ambiguous geometry groups. Explicit contradictions between an N vote and the same annotator's polygon are excluded.

We assess *visual predictability*, not actual repeat-annotation consistency. Read Sections 01, 03, 08 and 09 before Section 12. Output: `Outputs/v4/NBAA_annotation_consistency/12_simple_crown_models`.

In [ ]:
# 12A — Unified crown identity and annotation-source-aware labels.
from collections import defaultdict
import numpy as np, pandas as pd, geopandas as gpd
from IPython.display import display
needed=['ROOT','OUT','REF','POLY','matched','RGB_SOURCES','OVERVIEW_QA','UTM']
missing=[v for v in needed if v not in globals()]
assert not missing,('Run data setup Sections 00,01,03,08,09 first; missing: '+str(missing))
SIMPLE_OUT=OUT/'12_simple_crown_models'
SIMPLE_OUT.mkdir(parents=True,exist_ok=True)
PEOPLE=['Nadine','Thomas','Marijn','Joseph','Kristof']
SURVEYED={p:set(POLY[p].circle_id.astype(int)) if p in POLY else set() for p in PEOPLE}
RGB_BY_CIRCLE=OVERVIEW_QA.set_index('circle_id').source.to_dict()
records=[]
for r in REF.itertuples(index=False):
    records.append(dict(cid=int(r.circle_id),kind='ref',who=None,
         ref_id=str(r.verify_id),geom=r.geometry,key=str(r.verify_id)))
for person in POLY:
    for r in POLY[person].itertuples(index=False):
        records.append(dict(cid=int(r.circle_id),kind='poly',who=person,
             ref_id=None,geom=r.geometry,key=f'{person}:{r.crown_id}:{r.source_row}'))
parent=list(range(len(records)))
def root(i):
    while parent[i]!=i:
        parent[i]=parent[parent[i]]
        i=parent[i]
    return i
for i,a in enumerate(records):
    for j in range(i+1,len(records)):
        b=records[j]
        if a['cid']!=b['cid'] or not a['geom'].intersects(b['geom']):continue
        inter=a['geom'].intersection(b['geom']).area
        if inter<=0:continue
        iou=inter/max(a['geom'].area+b['geom'].area-inter,1e-8)
        smaller=inter/max(min(a['geom'].area,b['geom'].area),1e-8)
        if iou>=.10 or smaller>=.35:
            pa,pb=root(i),root(j)
            if pa!=pb:parent[pb]=pa
groups=defaultdict(list)
for i,r in enumerate(records):groups[root(i)].append(r)
rows=[];excluded=[]
for seq,group in enumerate(groups.values(),1):
    ref=[r for r in group if r['kind']=='ref']
    poly=[r for r in group if r['kind']=='poly']
    people=[r['who'] for r in poly]
    representative=ref[0] if ref else max(poly,key=lambda x:x['geom'].area)
    row=dict(object_id=f'C{group[0]["cid"]:02d}_O{seq:03d}',
      circle_id=int(group[0]['cid']),
      reference_id=ref[0]['ref_id'] if ref else None,
      owners=';'.join(sorted(set(people))),n_members=len(group),
      ambiguous=len(ref)>1 or len(people)!=len(set(people)),
      geometry=representative['geom'])
    (excluded if row['ambiguous'] else rows).append(row)
OBJECTS=gpd.GeoDataFrame(rows,geometry='geometry',crs=UTM)
OBJECTS.to_file(SIMPLE_OUT/'12_crown_objects.geojson',driver='GeoJSON')
pd.DataFrame([{k:v for k,v in row.items() if k!='geometry'} for row in excluded]).to_csv(
    SIMPLE_OUT/'12_ambiguous_exclusions.csv',index=False)
votes=matched.set_index('verify_id')
labels=[]
for r in OBJECTS.itertuples(index=False):
    owners=set(r.owners.split(';')) if r.owners else set()
    for p in PEOPLE:
        vote=votes.loc[r.reference_id,p] if pd.notna(r.reference_id) else None
        drew=p in owners
        surveyed=int(r.circle_id) in SURVEYED[p]
        if vote=='N' and drew:label=np.nan;source='conflict_excluded'
        elif drew:label=1;source='digitized_positive'
        elif vote=='M':label=1;source='reference_M'
        elif vote=='N':label=0;source='reference_N'
        elif vote=='U':label=np.nan;source='reference_U'
        elif surveyed:label=0;source='survey_assumed_negative'
        else:label=np.nan;source='not_assessed'
        labels.append(dict(object_id=r.object_id,annotator=p,circle_id=int(r.circle_id),
            target=label,source=source,reference_vote=vote))
LABELS=pd.DataFrame(labels)
LABELS.to_csv(SIMPLE_OUT/'12_object_annotator_labels.csv',index=False)
display(LABELS.groupby(['annotator','source']).size().unstack(fill_value=0))
display(LABELS.dropna(subset=['target']).groupby(['annotator','target']).size().unstack(fill_value=0))
print('Combined reference + polygon records:',len(records),
      'unique unambiguous crowns:',len(OBJECTS),'ambiguous excluded:',len(excluded))


## 13 — Frozen ResNet18 RGB crown image embeddings

Extract a 512-dimensional ImageNet ResNet18 descriptor from each original UAV RGB crown, **without training a neural network**. Each image patch is a square centred on the polygon, showing the crown and a little context. The pretrained encoder is independent of annotator labels. Only small raster windows are read; save embeddings to disk and reuse them. The polygons are used as crop locations rather than as segmentation targets.

In [ ]:
# 13A — CPU-only, memory-safe RGB crown embeddings; no U-Net or GPU needed.
import os, hashlib
import torch
from PIL import Image
from torchvision.models import resnet18, ResNet18_Weights
from rasterio.windows import Window, from_bounds
torch.set_num_threads(min(4,os.cpu_count() or 2))
EMBED_FILE=SIMPLE_OUT/'13_crown_rgb_resnet18.npz'
EMBED_INDEX=SIMPLE_OUT/'13_crown_embedding_index.csv'
digest=hashlib.sha256()
for r in OBJECTS.itertuples(index=False):
    digest.update(r.object_id.encode())
    digest.update(r.geometry.wkb)
    src=RGB_SOURCES[RGB_BY_CIRCLE[int(r.circle_id)]]
    st=src.stat()
    digest.update(str((src.name,st.st_size,st.st_mtime_ns)).encode())
digest.update(b'ImageNetResNet18V1_crown_margin1.7')
fingerprint=digest.hexdigest()
cached=False
if EMBED_FILE.is_file() and EMBED_INDEX.is_file():
    with np.load(EMBED_FILE,allow_pickle=False) as z:
        cached=str(z['fingerprint'].item())==fingerprint
        if cached: vectors=z['vectors'].copy()
    if cached:
        index=pd.read_csv(EMBED_INDEX)
        cached=len(index)==len(OBJECTS) and index.object_id.tolist()==OBJECTS.object_id.tolist()
if not cached:
    weights=ResNet18_Weights.IMAGENET1K_V1
    encoder=resnet18(weights=weights)
    encoder.fc=torch.nn.Identity()
    encoder.eval().cpu()
    preprocess=weights.transforms()
    vectors=[];meta=[]
    with torch.inference_mode():
        for i,r in enumerate(OBJECTS.itertuples(index=False)):
            name=RGB_BY_CIRCLE[int(r.circle_id)]
            with rasterio.open(RGB_SOURCES[name]) as ds:
                geom=gpd.GeoSeries([r.geometry],crs=UTM).to_crs(ds.crs).iloc[0]
                xmin,ymin,xmax,ymax=geom.bounds
                side=max(xmax-xmin,ymax-ymin)*1.7
                cx,cy=geom.centroid.x,geom.centroid.y
                win=from_bounds(cx-side/2,cy-side/2,cx+side/2,cy+side/2,
                                transform=ds.transform).round_offsets().round_lengths()
                try:
                    win=win.intersection(Window(0,0,ds.width,ds.height))
                except Exception:
                    win=None
                if win is None or win.width<4 or win.height<4:
                    patch=None
                else:
                    a=ds.read([1,2,3],window=win,masked=True).filled(0)
                    patch=np.moveaxis(np.uint8(np.clip(a,0,255)),0,-1)
            if patch is None or not patch.any():
                vectors.append(np.full(512,np.nan,dtype='float32'))
                meta.append(dict(object_id=r.object_id,valid_image=False,mosaic=name))
            else:
                x=preprocess(Image.fromarray(patch,'RGB')).unsqueeze(0)
                vectors.append(encoder(x)[0].numpy().astype('float32'))
                meta.append(dict(object_id=r.object_id,valid_image=True,mosaic=name))
            if (i+1)%25==0:print('Extracted',i+1,'/',len(OBJECTS),'crown images',flush=True)
    vectors=np.stack(vectors)
    index=pd.DataFrame(meta)
    np.savez_compressed(EMBED_FILE,fingerprint=np.array(fingerprint),vectors=vectors)
    index.to_csv(EMBED_INDEX,index=False)
    del encoder
assert vectors.shape==(len(OBJECTS),512)
VALID=set(index.loc[index.valid_image,'object_id'])
FEATURE_VECTORS=pd.DataFrame(vectors,index=OBJECTS.object_id)
print('Valid RGB crown image descriptors:',len(VALID),'/',len(OBJECTS))


## 14 — Simple per-annotator models and visual coherence

Compare a **regularized logistic regression** and a **small Random Forest**, both using the frozen ResNet18 crown image descriptors. Run five repeated 75/25 stratified crown-level train/test splits. Each spatially reconciled crown appears only once, so no duplicate polygon of a test crown enters training. Preprocessing is fitted only on training objects. Report sample sizes, AUROC, balanced accuracy and positive F1. These small-sample estimates are exploratory and do not quantify blind repeat-annotation reliability.

Also report positive-crown image similarity (median pairwise cosine distance), remembering that crown appearance legitimately changes with site and tree age.

In [ ]:
# 14A — Crown-level validation, no CNN training.
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.metrics.pairwise import cosine_distances

MODELS={
  'ResNet18 + logistic': lambda: Pipeline([
       ('scale',StandardScaler()),
       ('pca',PCA(n_components=0.95,svd_solver='full')),
       ('model',LogisticRegression(C=0.3,class_weight='balanced',max_iter=3000))]),
  'ResNet18 + Random Forest': lambda: RandomForestClassifier(
       n_estimators=200,max_depth=5,min_samples_leaf=2,
       class_weight='balanced',n_jobs=2,random_state=2026)
}
results=[]
for person in PEOPLE:
    selected=LABELS[(LABELS.annotator==person)&LABELS.target.notna()&
                    LABELS.object_id.isin(VALID)]
    counts=selected.target.value_counts().to_dict()
    if min(counts.get(0,0),counts.get(1,0))<4 or len(selected)<15:
        print('Skipped due to inadequate labelled positives/negatives:',person,counts)
        continue
    x=FEATURE_VECTORS.loc[selected.object_id].to_numpy(dtype='float32')
    y=selected.target.to_numpy(dtype=int)
    cv=StratifiedShuffleSplit(n_splits=5,test_size=.25,random_state=20261106)
    for repeat,(tr,te) in enumerate(cv.split(x,y),1):
        if len(set(y[te]))<2:continue
        for name,build in MODELS.items():
            model=build()
            model.fit(x[tr],y[tr])
            pred=model.predict(x[te])
            prob=model.predict_proba(x[te])[:,1]
            results.append(dict(annotator=person,model=name,repeat=repeat,
               crowns=len(y),M=int(y.sum()),N=int((1-y).sum()),test_crowns=len(te),
               auroc=roc_auc_score(y[te],prob),
               balanced_accuracy=balanced_accuracy_score(y[te],pred),
               positive_f1=f1_score(y[te],pred,zero_division=0)))
SCORES=pd.DataFrame(results)
assert len(SCORES),'No valid two-class experiments; inspect the table in Section 12A.'
SCORES.to_csv(SIMPLE_OUT/'14_repeated_classification_scores.csv',index=False)
summary=SCORES.groupby(['annotator','model']).agg(
    n_crowns=('crowns','first'),M=('M','first'),N=('N','first'),
    repeats=('repeat','count'),AUROC=('auroc','mean'),
    AUROC_sd=('auroc','std'),BA=('balanced_accuracy','mean'),
    BA_sd=('balanced_accuracy','std'),F1=('positive_f1','mean'),
    F1_sd=('positive_f1','std')).round(3)
summary.to_csv(SIMPLE_OUT/'14_model_summary.csv')
display(summary)
similarity=[]
for person in PEOPLE:
    positives=LABELS[(LABELS.annotator==person)&LABELS.target.eq(1)&
                     LABELS.object_id.isin(VALID)].object_id.unique()
    if len(positives)<3:continue
    X=FEATURE_VECTORS.loc[positives].to_numpy()
    distances=cosine_distances(X)
    vals=distances[np.triu_indices(len(X),1)]
    similarity.append(dict(annotator=person,n_M=len(X),
                           median_positive_cosine_distance=float(np.median(vals))))
SIMILARITY=pd.DataFrame(similarity)
SIMILARITY.to_csv(SIMPLE_OUT/'14_positive_image_similarity.csv',index=False)
display(SIMILARITY.round(3))
print('Visual predictability / feature similarity ≠ repeat-annotation consistency.')


## 15 — Interpretation

The original five-expert classification agreement is the direct inter-annotator comparison. These models provide a secondary check of how **visually learnable** each person's selections are, not who correctly identifies the species. True *intra-annotator repeatability* requires a blind second annotation round. Annotator datasets vary in crown count, survey coverage and positive prevalence; interpret model scores cautiously. The U-Net experiment has been removed from this notebook; historical U-Net outputs on Drive are not touched.

## 16 — All available field plots versus M2 U-Net (updated Data/Field GeoPackage)

**Authoritative field input:** `Final/Data/Field/Plot_Data_BA_nobuffer.gpkg`, with 109 mapped plots, their basal-area attributes and centres. The old 84-plot CSV and `Plot_Data.shp` are **not used**. No Nadine annotation circle filter is applied. This analysis samples the final-all30 M2 prediction mosaics directly, with correct UInt16 scaling (**0–10000; 65535 = NoData**).

**Geometry QA:** despite its name, some MOS and PK41 polygons in the new *nobuffer* layer remain oversized relative to the recorded `surface_ha`. The analysis derives a centred, north-aligned nominal square for each standard plot; checks the original versus census area; and excludes ForestGEO (aggregated 10 ha with incompatible 0.25-ha polygon) and Yoko `12Y1` (only three of four subplots inventoried, positions unknown). PK41 is reported only as a sensitivity test because its zero-Musanga inventory provenance differs from the 99-plot source CSV. Other mapped Yoko plots are included wherever M2 raster coverage permits. The five Yoko inventories without spatial geometry cannot be analysed until georeferenced.

**Outputs:** plot-by-plot audit of all 109 mapped records, full-primary and by-site statistics, optional PK41 sensitivity, and side-by-side scatterplots. A correlation between field **basal-area dominance** and predicted **canopy cover** is ecological association—not direct accuracy or pixel-segmentation skill.


In [ ]:
# 16A — All basal-area plots versus full M2 probability maps (no Nadine circle restriction)
import re, numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
from pathlib import Path
from shapely.geometry import box, mapping
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds, Window, transform as window_transform
from scipy.stats import linregress, spearmanr
import rasterio
from IPython.display import display

ROOT = globals().get('ROOT', Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'))
FIELD = ROOT/'Data'/'Field'
GPKG = FIELD/'Plot_Data_BA_nobuffer.gpkg'
MAPS = ROOT/'Outputs/V3/03_M2_full_mosaic_final_all30/probability_maps'
DEST = ROOT/'Outputs/v4/NBAA_annotation_consistency/16_field_BA_all_M2'
DEST.mkdir(parents=True,exist_ok=True)
MIN_VALID = 0.90
U16_SCALE = 10000.0  # UInt16 65535 = NoData, valid scores 0-10000

assert GPKG.exists(), f'Missing revised field dataset: {GPKG}'
assert MAPS.is_dir(), f'Missing M2 prediction folder: {MAPS}'

# Use THE NEW FIELD GEOPACKAGE directly. No 84-plot CSV join.
d = gpd.read_file(GPKG).to_crs('EPSG:32635')
required = {'Plot_name','Site','source','surface_ha','pct_BA_musanga'}
assert required.issubset(d.columns), f'GPKG lacks {required-set(d.columns)}'
assert d.Plot_name.is_unique, 'Plot identifiers must be unique'
assert d.geometry.notna().all()
print(f'Updated field plot polygons: {len(d)}')
print(d.groupby(['Site','source']).size().to_string())

d['gpkg_original_area_ha']=d.geometry.area/10000
d['geom_area_vs_inventory_ha']=d.gpkg_original_area_ha-d.surface_ha

# Explicit provenance and spatial-support flags.
# ForestGEO represents 10 ha of aggregated subquadrats, but is a
# 0.25-ha polygon here: exclude until its true 10-ha boundary is supplied.
# Yoko 12Y1 has only 3 of 4 subquadrats inventoried (0.12 ha); their
# locations are unknown. Don't substitute an arbitrary 0.12-ha square.
# PK41 BA=0 (15 plots) is not part of the 99-plot source CSV;
# keep as a separately reported sensitivity dataset.
d['exclusion_reason']=''
d.loc[d.source.eq('forgeo'),'exclusion_reason']='ForestGEO_10ha_boundary_unknown'
d.loc[(d.source.eq('yoko')) & d.Plot_name.str.upper().eq('12Y1'),
      'exclusion_reason']='Yoko_12Y1_3of4_subplots_unknown'
d.loc[d.source.eq('pk41'),'exclusion_reason']='PK41_separate_provenance_sensitivity'

# Reconstruct centred north-aligned, nominal-size squares, including
# incorrectly oversized MOS / PK41 polygons in supplied "nobuffer" GPKG.
# Retain original geometry for the flagged ForestGEO / partial-Yoko cases.
nominal=[]
for r in d.itertuples(index=False):
    if r.source=='forgeo' or (r.source=='yoko' and str(r.Plot_name).upper()=='12Y1'):
        nominal.append(r.geometry)
        continue
    half=np.sqrt(float(r.surface_ha)*10000.0)/2
    x,y=r.geometry.centroid.x,r.geometry.centroid.y
    nominal.append(box(x-half,y-half,x+half,y+half))
d=gpd.GeoDataFrame(d,geometry=gpd.GeoSeries(nominal,index=d.index,crs='EPSG:32635'))

d['field_site']=d.Site.replace({'Mos':'MOS'})
allowed={'Baego':('Baego_',), 'MOS':('Mos_',),
         'Yangambi':('Yangambi_',), 'Yoko':('Yoko_',)}
raster_paths=sorted(MAPS.glob('*_M2_final_all30_probability_u16_FULL.tif'))
assert raster_paths, 'No production M2 GeoTIFFs found'
by_site={site:[p for p in raster_paths if p.name.startswith(prefixes)]
         for site,prefixes in allowed.items()}
print('M2 mosaics found:',{k:len(v) for k,v in by_site.items()})

def read_one_plot(geom, filepath):
    with rasterio.open(filepath) as ds:
        if ds.count!=1 or ds.dtypes[0]!='uint16':
            raise ValueError('Unexpected model raster encoding: '+str(filepath))
        local=gpd.GeoSeries([geom],crs=d.crs).to_crs(ds.crs).iloc[0]
        footprint=box(*ds.bounds)
        intersection=local.intersection(footprint)
        if intersection.is_empty or intersection.area==0: return None
        # Limit window to intersection of field square with raster footprint.
        win=from_bounds(*intersection.bounds, transform=ds.transform)
        win=win.round_offsets().round_lengths()
        try: win=win.intersection(Window(0,0,ds.width,ds.height))
        except Exception: return None
        if win.width<1 or win.height<1: return None
        a=ds.read(1,window=win,masked=True)
        interior=geometry_mask([mapping(intersection)],out_shape=a.shape,
                               transform=window_transform(win,ds.transform),invert=True)
        raw=np.asarray(a.data)
        valid=interior & ~np.ma.getmaskarray(a) & (raw!=65535)
        nv=int(valid.sum())
        if not nv: return None
        vals=raw[valid].astype(np.float64)
        if vals.min()<0 or vals.max()>10000:
            raise ValueError(f'Probability values exceed 10000 in {filepath.name}; check scale')
        # Count valid pixel area / total nominal plot area, not just extent fraction.
        pixel_area=abs(ds.transform.a*ds.transform.e-ds.transform.b*ds.transform.d)
        coverage=min(1.,nv*pixel_area/local.area)
        return {'M2_cover_pct':float(vals.mean()/U16_SCALE*100),
                'raster_coverage':float(coverage), 'mosaic':filepath.name,
                'n_valid_pixels':nv, 'raw_q99':float(np.quantile(vals,.99))}

rows=[]
for r in d.itertuples(index=False):
    candidates=[]
    for path in by_site.get(r.field_site,[]):
        one=read_one_plot(r.geometry,path)
        if one is not None: candidates.append(one)
    result=max(candidates,key=lambda z:z['raster_coverage']) if candidates else {}
    rows.append({'Plot_name':r.Plot_name,**result})
measure=pd.DataFrame(rows)
d=d.merge(measure,on='Plot_name',how='left',validate='one_to_one')
d['raster_coverage']=d.raster_coverage.fillna(0)
d['included']=(d.raster_coverage>=MIN_VALID)&d.exclusion_reason.eq('')
d['plot_center_x']=d.geometry.centroid.x
d['plot_center_y']=d.geometry.centroid.y
audit=d.drop(columns='geometry')
audit.to_csv(DEST/'16_all_109_plot_audit.csv',index=False)
valid=d.loc[d.included & d.M2_cover_pct.notna()].copy()
print(f'Primary valid M2 cover for {len(valid)}/{len(d)} mapped field plots (minimum {MIN_VALID:.0%} raster coverage)')
print(pd.crosstab([d.field_site,d.source],d.included,dropna=False))
print('Excluded by protocol:', d.loc[d.exclusion_reason.ne(''),'exclusion_reason'].value_counts().to_dict())
display(audit[['Plot_name','source','surface_ha','gpkg_original_area_ha','pct_BA_musanga','M2_cover_pct','raster_coverage','mosaic','exclusion_reason','included']].round(2))
if len(valid)<3: raise ValueError('Too few plots with valid M2 pixels: check plot locations and GeoTIFF masks')

def stats_table(group, site):
    x=group.pct_BA_musanga.to_numpy(float)
    y=group.M2_cover_pct.to_numpy(float)
    out={'site':site,'n':len(x),'mean_field_BA_pct':float(np.mean(x)),
         'mean_M2_cover_pct':float(np.mean(y)),
         'mean_difference_pp':float(np.mean(y-x)),
         'MAE_to_BA_pp':float(np.mean(np.abs(y-x)))}
    if len(x)>2 and x.std()>0 and y.std()>0:
        fit=linregress(x,y)
        out.update(Pearson_r=fit.rvalue,Pearson_p=fit.pvalue,
                   Spearman_rho=spearmanr(x,y).statistic,
                   regression_R2=fit.rvalue**2,slope=fit.slope,intercept=fit.intercept)
    return out

stat=pd.DataFrame([stats_table(valid,'ALL')]+[
    stats_table(sub,site) for site,sub in valid.groupby('field_site')])
display(stat.round(3))
stat.to_csv(DEST/'16_M2_vs_field_BA_stats.csv',index=False)
# PK41 sensitivity: calculate separately, not in pooled primary validation.
pk41=d.loc[d.source.eq('pk41') & (d.raster_coverage>=MIN_VALID) & d.M2_cover_pct.notna()]
if len(pk41):
    sensitivity=pd.DataFrame([stats_table(pk41,'PK41_ONLY'),stats_table(pd.concat([valid,pk41]),'PRIMARY_PLUS_PK41')])
    display(sensitivity.round(3))
    sensitivity.to_csv(DEST/'16_PK41_sensitivity.csv',index=False)

palette={s:plt.cm.tab10(i) for i,s in enumerate(sorted(valid.field_site.unique()))}
fig,axes=plt.subplots(1,2,figsize=(13,5.6),layout='constrained')
ax=axes[0]
for site,sub in valid.groupby('field_site'):
    ax.scatter(sub.pct_BA_musanga,sub.M2_cover_pct,s=55,alpha=.85,
               edgecolor='white',linewidth=.6,label=f'{site} (n={len(sub)})',
               color=palette[site])
ax.plot([0,100],[0,100],'--',lw=1,color='gray')
if valid.pct_BA_musanga.std()>0 and valid.M2_cover_pct.std()>0:
    fit=linregress(valid.pct_BA_musanga,valid.M2_cover_pct)
    ax.plot([0,100],[fit.intercept,fit.intercept+100*fit.slope],color='black',lw=1.5)
    ax.text(.03,.97,f'n={len(valid)}  r={fit.rvalue:.3f}  R²={fit.rvalue**2:.3f}\nslope={fit.slope:.2f}  intercept={fit.intercept:.1f}',
            transform=ax.transAxes,va='top',fontsize=10)
ax.set(xlabel='Observed Musanga basal area (%)',ylabel='M2 U-Net canopy cover (%)',
       title='All valid field plots — pooled',xlim=(0,100),ylim=(0,100))
ax.grid(alpha=.2)
ax.legend(loc='lower right',fontsize=8)
ax=axes[1]
for site,sub in valid.groupby('field_site'):
    ax.scatter(sub.pct_BA_musanga,sub.M2_cover_pct,s=55,alpha=.85,
               edgecolor='white',linewidth=.6,color=palette[site],
               label=f'{site} (n={len(sub)})')
    if len(sub)>2 and sub.pct_BA_musanga.std()>0:
        fit=linregress(sub.pct_BA_musanga,sub.M2_cover_pct)
        xx=np.linspace(sub.pct_BA_musanga.min(),sub.pct_BA_musanga.max(),30)
        ax.plot(xx,fit.intercept+fit.slope*xx,color=palette[site],lw=1.7)
ax.plot([0,100],[0,100],'--',lw=1,color='gray')
ax.set(xlabel='Observed Musanga basal area (%)',ylabel='M2 U-Net canopy cover (%)',
       title='Within-site fits',xlim=(0,100),ylim=(0,100))
ax.grid(alpha=.2)
ax.legend(fontsize=8)
fig.suptitle('Independent field Musanga BA dominance vs V3_1 M2 cover (all available valid plots)')
fig.savefig(DEST/'16_all_plots_M2_vs_BA.png',dpi=240,bbox_inches='tight')
fig.savefig(DEST/'16_all_plots_M2_vs_BA.pdf',bbox_inches='tight')
plt.show()
print('Saved plot-level audit, statistics and figures:', DEST)
print('CAVEAT: nominal centred squares for standard plots; ForestGEO and Yoko 12Y1 excluded.')
